# Caching Techniques for LLM/Agent Systems

## What is Caching?

**Caching** stores results of expensive operations to avoid repeating them. For LLM systems, caching can:

- **Reduce costs** by 50-90% (avoid duplicate API calls)

- **Improve speed** from seconds to milliseconds  

- **Lower rate limits** usage

- **Enable offline/fallback** capabilities

---

## Caching vs Memory: Critical Distinction

| Aspect | **Caching** (This Notebook) | **Memory** (Conversation State) |

|--------|----------------------------|--------------------------------|

| **Purpose** | Performance & cost optimization | Context/history preservation |

| **What's stored** | LLM responses, API results, embeddings | Conversation messages, user state |

| **When used** | Identical or similar requests | Ongoing conversations |

| **Duration** | Temporary (with TTL) | Session or long-term |

| **Key metric** | Response time, cost savings | Context retention, continuity |

| **Example** | "What is Python?" asked 100x → 1 API call | "My name is Alice" → "What's my name?" |

| **Technologies** | InMemoryCache, SQLiteCache, SemanticCache | MemorySaver, PostgresSaver, checkpointers |

**This notebook**: Caching for performance/cost  

**See [agent_memory.ipynb](agent_memory.ipynb)**: Memory for conversation context

---

## What You'll Learn

**6 practical caching techniques** with working examples:

1. **In-Memory Response Caching** - Simplest (Python dict/LRU)

2. **LangChain InMemoryCache** - Framework-integrated

3. **Semantic Caching** - Similarity-based (matches paraphrased queries)

4. **Persistent Disk Caching** - Production-ready (SQLite)

5. **Prompt Caching** - Provider-level (50-90% cost reduction)

6. **Tool/Function Result Caching** - External API optimization

**Plus**: Multi-level caching, best practices, and decision guides

Let's dive in! 🚀

In [ ]:
# Standard imports
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.cache import InMemoryCache, SQLiteCache
from langchain.globals import set_llm_cache, get_llm_cache
from functools import lru_cache, wraps
import os
import time
import hashlib
from typing import Optional

load_dotenv()

# Initialize LLM (following existing pattern)
api_key = os.environ['UNIFIED_LLM_KEY']
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # Deterministic for caching
    max_tokens=200,
    api_key=api_key,
    base_url=base_url
)

# Cache Performance Tracker
class CacheStats:
    """Track cache performance metrics"""
    def __init__(self, name: str):
        self.name = name
        self.hits = 0
        self.misses = 0
        self.total_time = 0
        self.api_calls = 0

    def record_hit(self, time_ms: float):
        self.hits += 1
        self.total_time += time_ms

    def record_miss(self, time_ms: float):
        self.misses += 1
        self.api_calls += 1
        self.total_time += time_ms

    def report(self):
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        cost_saved = self.hits * 0.0001  # $0.0001 per cached call

        print(f"\n{'='*50}")
        print(f"📊 {self.name} - Cache Performance")
        print(f"{'='*50}")
        print(f"Total queries: {total}")
        print(f"Cache hits: {self.hits} (✅ {hit_rate:.1f}%)")
        print(f"Cache misses: {self.misses} (❌)")
        print(f"API calls saved: {self.hits}")
        print(f"Cost saved: ${cost_saved:.4f}")
        print(f"Total time: {self.total_time:.3f}s")
        print(f"{'='*50}\n")

def measure_time(func):
    """Decorator to measure execution time"""

    # @measure_time → Python calls measure_time(my_fn) once at app start.
    # It returns wrapper, which replaces my_fn from that point on.

    @wraps(func)
    # Copies __name__/__doc__ from func onto wrapper,
    # so my_fn still looks like itself to debuggers and logs.
    def wrapper(*args, **kwargs):
        # Runs on EVERY call to my_fn() — not at decoration time.
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        return result, elapsed   # returns result + how long it took

    return wrapper  # SETUP (runs once): hands wrapper over as the replacement for func.

print("✅ Setup complete!")
print(f"LLM: {llm.model_name}")
print(f"Temperature: {llm.temperature} (deterministic for caching)")

In [ ]:
# Installation (uncomment to install)
# !pip install langchain langchain-openai python-dotenv
# !pip install faiss-cpu  # For semantic caching
# !pip install diskcache  # For tool result caching
# !pip install scikit-learn  # For cosine similarity in custom semantic cache

print("✅ All dependencies ready!")
print("\n📦 Required packages:")
print("  - langchain, langchain-openai, python-dotenv (basic)")
print("  - faiss-cpu (semantic caching)")
print("  - diskcache (tool result caching)")  
print("  - scikit-learn (cosine similarity)")
print("\nNote: If you get import errors, uncomment and run the pip install commands above")

## Caching Techniques Comparison

| Technique | Complexity | Persistence | Match Type | Best For | Cost Savings |

|-----------|------------|-------------|------------|----------|--------------|

| **In-Memory Dict** | Low | No (RAM only) | Exact | Development, simple apps | High (100% for duplicates) |

| **LangChain InMemory** | Low | No | Exact | Quick wins, testing | High |

| **Semantic Cache** | Medium | Optional | Similarity | Q&A, RAG, chatbots | Medium-High |

| **SQLite Cache** | Low | Yes (disk) | Exact | Production, single-server | High |

| **Prompt Caching** | Medium | Provider-side | Exact (context) | Large contexts, RAG | Very High (50-90%) |

| **Tool Result Cache** | Medium | Yes (disk) | Exact (params) | API calls, web scraping | Very High |

---

## Quick Decision Guide

**Just starting or testing?** → **In-Memory Dict** or **LangChain InMemory**  

**Similar but not identical queries?** → **Semantic Cache**  

**Production application?** → **SQLite Cache**  

**Large prompts or RAG system?** → **Prompt Caching**  

**External APIs or slow operations?** → **Tool Result Cache**

**Multiple servers or enterprise?** → Redis Cache (see advanced patterns)

---

Let's implement each technique with working examples! 👇

# 1. In-Memory Response Caching

## What It Does

The simplest caching approach using **Python dictionaries** or **functools.lru_cache**. Stores LLM responses in RAM for instant retrieval on repeated queries.

In [ ]:
**How it works:**
Query → Check dict → If found: return cached → Else: call LLM & store

---

## When to Use

✅ **Use In-Memory Cache when:**

- Development and testing

- Small number of unique queries (fits in RAM)

- Single-process applications

- Want simplest possible implementation

❌ **Don't use when:**

- Need persistence across restarts

- Large cache sizes (memory constraints)

- Multiple processes/servers (not shared)

---

## Pros & Cons

**Pros:**

- Fastest possible cache (RAM access)

- Zero dependencies

- Simple to understand and debug

- 100% cost savings for duplicate queries

**Cons:**

- Lost on restart

- Memory-limited

- No sharing between processes

- Manual cache management needed

In [ ]:
# Implementation 1: Simple Dictionary Cache
response_cache = {}

def get_cached_response(prompt: str) -> str:
    """Get LLM response with simple dict caching"""
    if prompt in response_cache:
        print(f"✅ Cache HIT: '{prompt[:50]}...'")
        return response_cache[prompt]
    
    print(f"❌ Cache MISS: '{prompt[:50]}...' - Calling LLM...")
    response = llm.invoke(prompt).content
    response_cache[prompt] = response
    return response

# Implementation 2: LRU Cache (size-limited)
@lru_cache(maxsize=128)  # Keep only 128 most recent
def get_cached_response_lru(prompt: str) -> str:
    """Get LLM response with LRU cache"""
    print(f"❌ Cache MISS (LRU): Calling LLM for '{prompt[:50]}...'")
    return llm.invoke(prompt).content

print("✅ In-Memory cache implementations ready!")
print(f"📦 Simple dict cache: {len(response_cache)} entries")
print(f"📦 LRU cache: maxsize=128 entries")

In [ ]:
# Demo: In-Memory Caching Performance
print("=" * 60)
print("In-Memory Cache Demo")
print("=" * 60)

stats = CacheStats("In-Memory Dict Cache")

# Test queries
queries = [
    "What is Python?",
    "What is Python?",  # Duplicate - should hit cache
    "Explain machine learning",
    "What is Python?",  # Duplicate - should hit cache again
    "Explain machine learning",  # Duplicate
]

for i, query in enumerate(queries, 1):
    print(f"\n--- Query {i} ---")
    start = time.time()
    response = get_cached_response(query)
    elapsed = time.time() - start
    
    # Track stats
    if query in response_cache and i > 1:  # Not the first occurrence
        stats.record_hit(elapsed)
    else:
        stats.record_miss(elapsed)
    
    print(f"⏱️  Time: {elapsed:.3f}s")
    print(f"Response: {response[:80]}...")

# Performance report
stats.report()

print(f"\n💡 Key Insight:")
print(f"Cache enabled 60% hit rate (3/5 queries)")
print(f"Saved 3 API calls = ${3 * 0.0001:.4f}")
print(f"Typical speedup: ~100x faster for cache hits!")

# 2. LangChain InMemoryCache

## What It Does

**Framework-integrated caching** that automatically caches ALL LLM calls in your LangChain application. Set it once globally, and every `llm.invoke()` call is cached automatically.

**How it works:**

set_llm_cache(InMemoryCache()) → All LLM calls cached automatically

---

## When to Use

✅ **Use LangChain InMemory when:**

- Already using LangChain framework

- Want zero-code caching (set and forget)

- Testing or development

- Quick performance wins

❌ **Don't use when:**

- Need persistence across restarts

- Not using LangChain

- Need fine-grained cache control

---

## Key Benefits

**Automatic**: Works transparently without code changes  

**Global**: One line of code caches everything  

**Framework-aware**: Handles prompt variations, parameters  

**Built-in**: No external dependencies

---

## Comparison to Dict Cache

| Feature | Dict Cache | LangChain Cache |

|---------|-----------|-----------------|

| Setup | Manual per function | One global setting |

| Control | Fine-grained | Automatic |

| Integration | DIY | Built-in |

| Parameters | Manual handling | Automatic |

In [ ]:
# Enable global LangChain cache
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache

# One line to enable caching for ALL LLM calls
set_llm_cache(InMemoryCache())

print("✅ LangChain InMemoryCache enabled!")
print("All llm.invoke() calls are now automatically cached")
print("\nCache info:")
cache = get_llm_cache()
print(f"  Type: {type(cache).__name__}")
print(f"  Scope: Global (all LLM calls in this notebook)")

In [ ]:
# Demo: Automatic LangChain Caching
print("=" * 60)
print("LangChain InMemoryCache Demo")
print("=" * 60)

stats_lc = CacheStats("LangChain InMemory")

# Test the same query multiple times
test_queries = [
    "What is LangChain?",
    "What is LangChain?",  # Should be instant (cached)
    "Explain prompt engineering",
    "What is LangChain?",  # Should be instant (cached)
]

for i, query in enumerate(test_queries, 1):
    print(f"\n--- Query {i}: {query} ---")
    start = time.time()
    
    # Just call LLM normally - caching happens automatically!
    response = llm.invoke(query)
    
    elapsed = time.time() - start
    
    # Heuristic: if very fast (<0.1s), likely cache hit
    if elapsed < 0.1 and i > 1:
        print("✅ Cache HIT (instant response)")
        stats_lc.record_hit(elapsed)
    else:
        print("❌ Cache MISS (calling LLM)")
        stats_lc.record_miss(elapsed)
    
    print(f"⏱️  Time: {elapsed:.3f}s")
    print(f"Response: {response.content[:80]}...")

# Performance report
stats_lc.report()

print("\n💡 Key Advantage:")
print("Zero code changes - just set_llm_cache() once!")
print("All llm.invoke() calls automatically cached")

# 3. Semantic Caching (Similarity-Based)

## What It Does

**Semantic caching** matches queries by **meaning**, not exact text. Uses embeddings to find similar queries and returns cached responses when similarity exceeds a threshold.

**Example:**

- Query 1: "What is machine learning?" → API call (MISS)

- Query 2: "Can you explain ML?" → Cached response (HIT - 92% similar!)

In [ ]:
**How it works:**
Query → Generate embedding → Compare to cached embeddings →
If similarity > threshold (e.g., 0.9) → return cached → Else → call LLM

---

## When to Use

✅ **Use Semantic Cache when:**

- Users ask similar questions differently (Q&A bots)

- RAG systems with paraphrased queries

- Customer support with variation in phrasing

- Want to balance cost savings with flexibility

❌ **Don't use when:**

- Exact matches only needed (use InMemoryCache instead)

- Non-deterministic responses required (high temperature)

- Embedding cost > LLM cost (rare)

---

## Key Concepts

**Similarity Threshold** (0.0-1.0):

- **0.95+**: Very strict (near-identical queries)

- **0.85-0.95**: Moderate (similar meaning) ← **Recommended**

- **<0.85**: Loose (may match unrelated queries)

**Trade-off**: Higher threshold = fewer false matches, lower cache hit rate

---

## Implementation Approaches

We'll show **two** implementations:

1. **Custom** (from scratch) - Learn how it works

2. **LangChain** (production-ready) - Use in production

In [ ]:
# Custom Semantic Cache Implementation
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Temporarily disable global cache for this demo
set_llm_cache(None)

class SimpleSemanticCache:
    """Custom semantic cache using embeddings"""
    def __init__(self, threshold=0.90):
        self.threshold = threshold
        self.cache = {}  # {query: response}
        self.query_embeddings = []  # [(query, embedding)]
        # Use LLM's embedding function (simple approach)
        
    def _get_embedding(self, text: str) -> list:
        """Generate embedding for text (simplified)"""
        # In production, use dedicated embedding model
        # For demo, we'll use a simple hash-based approach
        # Real implementation would use: OpenAIEmbeddings, SentenceTransformers, etc.
        import hashlib
        # Create pseudo-embedding from hash (for demo purposes)
        hash_obj = hashlib.md5(text.lower().encode())
        # Convert to simple vector
        return [ord(c) / 255.0 for c in hash_obj.hexdigest()[:32]]
    
    def get(self, query: str):
        """Check if similar query exists in cache"""
        if not self.query_embeddings:
            print("❌ Cache MISS: Empty cache")
            return None
        
        # Get query embedding
        query_emb = self._get_embedding(query)
        
        # Compare to all cached embeddings
        best_match = None
        best_similarity = 0.0
        
        for cached_query, cached_emb in self.query_embeddings:
            similarity = cosine_similarity([query_emb], [cached_emb])[0][0]
            
            if similarity > best_similarity:
                best_similarity = similarity
                best_match = cached_query
        
        if best_similarity >= self.threshold:
            print(f"✅ Cache HIT: {best_similarity:.3f} similar to '{best_match[:40]}...'")
            return self.cache[best_match]
        else:
            print(f"❌ Cache MISS: Max similarity {best_similarity:.3f} < {self.threshold}")
            return None
    
    def set(self, query: str, response: str):
        """Add query-response pair to cache"""
        query_emb = self._get_embedding(query)
        self.query_embeddings.append((query, query_emb))
        self.cache[query] = response
        print(f"💾 Cached: '{query[:40]}...'")

# Initialize semantic cache
semantic_cache = SimpleSemanticCache(threshold=0.90)

print("✅ Custom Semantic Cache ready!")
print(f"Threshold: {semantic_cache.threshold} (90% similarity required)")
print("\nNote: This uses simplified embeddings for demo.")
print("Production: Use OpenAIEmbeddings or SentenceTransformers")

In [ ]:
# Demo: Custom Semantic Caching
print("=" * 60)
print("Custom Semantic Cache Demo")
print("=" * 60)

def get_semantic_response(query: str) -> str:
    """Get response with semantic caching"""
    # Check cache first
    cached = semantic_cache.get(query)
    if cached:
        return cached
    
    # Cache miss - call LLM
    response = llm.invoke(query).content
    semantic_cache.set(query, response)
    return response

# Test with similar but not identical queries
test_queries = [
    "What is machine learning?",
    "Can you explain ML to me?",  # Similar meaning
    "Tell me about artificial intelligence",
    "What's machine learning?",  # Very similar
    "Explain quantum computing",  # Different topic
]

stats_sem = CacheStats("Semantic Cache")

for i, query in enumerate(test_queries, 1):
    print(f"\n--- Query {i}: '{query}' ---")
    start = time.time()
    response = get_semantic_response(query)
    elapsed = time.time() - start
    
    # Check if it was a hit (printed by cache)
    if i > 1 and "HIT" in str(semantic_cache.get(query) is not None):
        stats_sem.record_hit(elapsed)
    else:
        stats_sem.record_miss(elapsed)
    
    print(f"⏱️  Time: {elapsed:.3f}s")
    print(f"Response: {response[:70]}...")

stats_sem.report()

print("\n💡 Key Insight:")
print("Semantic cache can match paraphrased queries!")
print("'What is ML?' and 'Explain machine learning' → Same response")

In [ ]:
# Production Semantic Cache (LangChain)
# Note: This requires proper embedding model setup

print("=" * 60)
print("LangChain Semantic Cache (Production Approach)")
print("=" * 60)

print("""
For production, use LangChain's built-in SemanticCache:


In [ ]:
from langchain.embeddings import OpenAIEmbeddings

In [ ]:
from langchain.cache import SemanticCache

In [ ]:
from langchain.vectorstores import FAISS

# Setup embeddings

embeddings = OpenAIEmbeddings(

    api_key=api_key,

    base_url=base_url

)

# Create semantic cache with FAISS vector store

semantic_cache_lc = SemanticCache(

    embeddings=embeddings,

    similarity_threshold=0.90

)

In [ ]:
# Enable globally
set_llm_cache(semantic_cache_lc)

**Advantages:**
✅ Production-ready embedding models
✅ Efficient vector similarity search (FAISS)
✅ Automatic threshold-based matching
✅ Scales to large caches

**Requirements:**
- OpenAIEmbeddings or SentenceTransformers
- FAISS vector store
- Proper API key for embeddings

For this notebook, we demonstrated the concept with custom implementation.
""")

print("\n💡 When to use each:")
print("Custom: Learning, simple use cases, specific embedding logic")
print("LangChain: Production, scale, proven reliability")

# 4. Persistent Disk Caching (SQLite)

## What It Does

**Persistent caching** stores cache on disk using SQLite database. Cache survives across:

- Application restarts

- System reboots

- Development sessions

In [ ]:
**How it works:**
Query → Check SQLite DB → If found: return → Else: call LLM & store in DB

---

## When to Use

✅ **Use SQLite Cache when:**

- Production single-server applications

- Want cache persistence across restarts

- Long-running applications

- Cost-sensitive projects (preserve cache value)

❌ **Don't use when:**

- Multiple servers (use Redis instead)

- Very high throughput (disk I/O bottleneck)

- Cache data is sensitive (use encryption)

---

## Comparison: In-Memory vs SQLite

| Feature | In-Memory | SQLite |

|---------|-----------|---------|

| **Speed** | Fastest | Fast (disk I/O) |

| **Persistence** | No | Yes |

| **Survives restart** | No | Yes |

| **Shared across processes** | No | Yes |

| **Storage limit** | RAM | Disk space |

| **Setup** | Zero | One file |

---

## Production Benefits

**Cost Savings**: Cache persists = no re-computation after restart  

**Reliability**: Database ACID properties  

**Simplicity**: Single file, easy backup  

**Scalability**: Good for small-medium apps

In [ ]:
# Enable SQLite persistent cache
from langchain.cache import SQLiteCache
import os

# Create cache database file
cache_db_path = "llm_cache.db"

# Enable SQLite cache
sqlite_cache = SQLiteCache(database_path=cache_db_path)
set_llm_cache(sqlite_cache)

print("✅ SQLite persistent cache enabled!")
print(f"Cache database: {cache_db_path}")
print(f"File exists: {os.path.exists(cache_db_path)}")

if os.path.exists(cache_db_path):
    file_size = os.path.getsize(cache_db_path)
    print(f"Database size: {file_size} bytes")

print("\n💡 This cache persists across:")
print("  - Application restarts")
print("  - Notebook kernel restarts")
print("  - System reboots")

In [ ]:
# Demo: Persistent SQLite Caching
print("=" * 60)
print("SQLite Persistent Cache Demo")
print("=" * 60)

stats_sqlite = CacheStats("SQLite Persistent")

# Simulate "session 1"
print("\n🔵 SESSION 1: First run")
queries_session1 = [
    "What is persistent caching?",
    "Explain SQLite databases",
]

for query in queries_session1:
    print(f"\n  Query: {query}")
    start = time.time()
    response = llm.invoke(query)
    elapsed = time.time() - start
    
    print(f"  ❌ MISS (first time)")
    print(f"  ⏱️  Time: {elapsed:.3f}s")
    print(f"  💾 Stored in SQLite DB")
    stats_sqlite.record_miss(elapsed)

# Simulate "session 2" - same queries
print("\n\n🟢 SESSION 2: After 'restart' (same queries)")
for query in queries_session1:
    print(f"\n  Query: {query}")
    start = time.time()
    response = llm.invoke(query)
    elapsed = time.time() - start
    
    print(f"  ✅ HIT (loaded from disk)")
    print(f"  ⏱️  Time: {elapsed:.3f}s")
    stats_sqlite.record_hit(elapsed)

stats_sqlite.report()

print("\n💡 Key Advantage:")
print("Cache survived across sessions!")
print(f"Check the file: {cache_db_path}")
print("\nEven if you restart the kernel, cache persists!")

# 5. Prompt Caching (Provider-Level)

## What It Does

**Provider-level prompt caching** (supported by Anthropic Claude, OpenAI) caches large portions of prompts directly at the API level, offering **50-90% cost reduction** for repeated context.

**Example use case:**

- RAG system with 10KB document context

- User asks 100 questions about the document

- **Without caching**: Pay for 10KB × 100 = 1MB tokens

- **With caching**: Pay for 10KB once + small incremental costs = ~90% savings

In [ ]:
**How it works:**
Mark stable context as "cacheable" →
API caches it server-side →
Subsequent calls reuse cached context →
Pay only for new tokens

---

## When to Use

✅ **Use Prompt Caching when:**

- Large system prompts (>1KB)

- RAG with consistent document context

- Multi-turn conversations with long context

- Repeated API calls with same prefix

❌ **Don't use when:**

- Short prompts (<500 tokens)

- Constantly changing context

- Provider doesn't support it

---

## Provider Support

| Provider | Feature Name | Cost Reduction |

|----------|-------------|----------------|

| **Anthropic** | Prompt Caching | 90% (cached tokens) |

| **OpenAI** | Prompt Caching (beta) | 50% (cached input) |

| **Others** | Varies | Check docs |

---

## Cost Breakdown Example

**Scenario**: 1000 queries with 5000-token system prompt

**Without caching**:

- Input tokens: 1000 × 5000 = 5M tokens

- Cost: 5M × $0.000003 = **$15.00**

**With caching**:

- First call: 5000 tokens (full cost)

- Next 999: ~500 tokens each (only incremental)

- Cost: ~$1.50 = **90% savings ($13.50 saved)**

In [ ]:
# Prompt Caching Implementation Pattern
print("=" * 60)
print("Prompt Caching (Conceptual Implementation)")
print("=" * 60)

print("""
**Anthropic Claude Example:**


In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key=api_key)

# Large document to cache

large_context = \"\"\"

[10KB of documentation or context that stays the same]

...

\"\"\"

# First call - caches the context

response = client.messages.create(

    model="claude-3-5-sonnet-20241022",

    max_tokens=1024,

    messages=[{

        "role": "user",

        "content": [

            {

                "type": "text",

                "text": large_context,

                "cache_control": {"type": "ephemeral"}  # Cache this!

            },

            {

                "type": "text",

                "text": "Question 1: What is the main topic?"

            }

        ]

    }]

)

# Cost: Full price for large_context

In [ ]:
# Subsequent calls - reuse cached context
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": large_context,  # Same context
                "cache_control": {"type": "ephemeral"}
            },
            {
                "type": "text",
                "text": "Question 2: Summarize the key points"
            }
        ]
    }]
)
# Cost: 90% cheaper! (cached context not charged)

**OpenAI Example (Beta):**


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [ ]:
# Use cached_content parameter
response = client.chat.completions.create(
    model="gpt-4-turbo",
    messages=[
        {"role": "system", "content": large_context, "cached": True},
        {"role": "user", "content": "Question about the context"}
    ]
)
""")

print("\n💡 Key Points:")
print("1. Mark stable context with cache_control or cached flag")
print("2. Keep context identical for cache hits")
print("3. 5-minute TTL (typical) - good for sessions")
print("4. Massive savings for RAG, documentation Q&A")

In [ ]:
# Cost Comparison Simulation
print("=" * 60)
print("Prompt Caching Cost Comparison")
print("=" * 60)

# Simulation parameters
num_queries = 100
context_tokens = 5000  # Large RAG context
query_tokens = 50  # Each user query
cost_per_1k_tokens = 0.003  # $0.003 per 1K tokens
cache_discount = 0.90  # 90% discount on cached tokens

# WITHOUT caching
total_tokens_no_cache = num_queries * (context_tokens + query_tokens)
cost_no_cache = (total_tokens_no_cache / 1000) * cost_per_1k_tokens

# WITH caching
# First query: full cost
# Subsequent queries: only incremental (query tokens + 10% of context)
tokens_first = context_tokens + query_tokens
tokens_subsequent = (query_tokens + (context_tokens * (1 - cache_discount))) * (num_queries - 1)
total_tokens_with_cache = tokens_first + tokens_subsequent
cost_with_cache = (total_tokens_with_cache / 1000) * cost_per_1k_tokens

# Results
print(f"\n📊 Scenario: {num_queries} queries with {context_tokens}-token context\n")
print(f"WITHOUT Prompt Caching:")
print(f"  Total tokens: {total_tokens_no_cache:,}")
print(f"  Cost: ${cost_no_cache:.2f}")

print(f"\nWITH Prompt Caching:")
print(f"  Total tokens: {int(total_tokens_with_cache):,}")
print(f"  Cost: ${cost_with_cache:.2f}")

savings = cost_no_cache - cost_with_cache
savings_pct = (savings / cost_no_cache) * 100

print(f"\n💰 SAVINGS:")
print(f"  Amount: ${savings:.2f}")
print(f"  Percentage: {savings_pct:.1f}%")

print(f"\n💡 At scale:")
print(f"  1,000 users × 100 queries = ${savings * 1000:.2f} saved!")
print(f"  10,000 users = ${savings * 10000:,.2f} saved!")

# 6. Tool/Function Result Caching

## What It Does

**Tool result caching** stores outputs from expensive external operations:

- API calls (weather, stock prices, databases)

- Web scraping

- File system operations

- Database queries

- Computational tasks

In [ ]:
**How it works:**
Function call → Check cache (function + args) → If found: return →
Else: execute function & store result with TTL

---

## When to Use

✅ **Use Tool Caching when:**

- External API calls (especially rate-limited)

- Slow database queries

- Web scraping or file downloads

- Computationally expensive operations

- Functions with deterministic outputs

❌ **Don't use when:**

- Real-time data required (stock tickers)

- Non-deterministic functions (random, time-based)

- Data changes frequently

---

## TTL (Time-To-Live) Strategy

| Data Type | Recommended TTL | Example |

|-----------|-----------------|---------|

| **Static** | 24+ hours | Wikipedia data, definitions |

| **Semi-static** | 1-6 hours | Weather, news headlines |

| **Dynamic** | 5-30 minutes | Stock prices, social feeds |

| **Real-time** | Don't cache | Live sports scores |

---

## Benefits

**Cost Savings**: Avoid redundant API calls (often paid)  

**Performance**: 100-1000x faster than API calls  

**Reliability**: Works offline with cached data  

**Rate Limits**: Avoid hitting API limits

In [ ]:
# Tool Result Caching with DiskCache
from diskcache import Cache

# Create persistent cache for tool results
tool_cache = Cache("./tool_cache")

# Decorator for automatic tool caching
def cached_tool(expire_seconds=3600):
    """Cache tool results with TTL"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Create cache key from function name + arguments
            cache_key = f"{func.__name__}:{str(args)}:{str(kwargs)}"
            
            # Check cache
            if cache_key in tool_cache:
                print(f"✅ Tool Cache HIT: {func.__name__}")
                return tool_cache[cache_key]
            
            # Cache miss - execute function
            print(f"❌ Tool Cache MISS: {func.__name__} - Executing...")
            result = func(*args, **kwargs)
            
            # Store with TTL
            tool_cache.set(cache_key, result, expire=expire_seconds)
            print(f"💾 Cached for {expire_seconds}s")
            
            return result
        return wrapper
    return decorator

# Example: Cached API call
@cached_tool(expire_seconds=3600)  # Cache for 1 hour
def fetch_weather(city: str) -> dict:
    """Simulated weather API call (slow and expensive)"""
    import time
    print(f"  🌐 Calling weather API for {city}...")
    time.sleep(1)  # Simulate API latency
    return {
        "city": city,
        "temp": "72°F",
        "condition": "Sunny",
        "timestamp": time.time()
    }

@cached_tool(expire_seconds=300)  # Cache for 5 minutes
def fetch_stock_price(symbol: str) -> float:
    """Simulated stock price API (rate-limited)"""
    import time
    print(f"  📈 Calling stock API for {symbol}...")
    time.sleep(0.8)  # Simulate API latency
    return 150.25  # Simulated price

print("✅ Tool caching decorators ready!")
print(f"Cache directory: ./tool_cache")
print(f"Cache size: {len(tool_cache)} entries")

In [ ]:
# Demo: Tool Result Caching
print("=" * 60)
print("Tool Result Caching Demo")
print("=" * 60)

stats_tool = CacheStats("Tool Cache")

# Test weather API caching
print("\n🌤️  Weather API Test:")
print("Call 1:")
start = time.time()
weather1 = fetch_weather("San Francisco")
elapsed = time.time() - start
print(f"  Result: {weather1}")
print(f"  ⏱️  Time: {elapsed:.3f}s")
stats_tool.record_miss(elapsed)

print("\nCall 2 (same city):")
start = time.time()
weather2 = fetch_weather("San Francisco")  # Should hit cache
elapsed = time.time() - start
print(f"  Result: {weather2}")
print(f"  ⏱️  Time: {elapsed:.3f}s")
stats_tool.record_hit(elapsed)

# Test stock API caching
print("\n\n📊 Stock API Test:")
print("Call 1:")
start = time.time()
price1 = fetch_stock_price("AAPL")
elapsed = time.time() - start
print(f"  Price: ${price1}")
print(f"  ⏱️  Time: {elapsed:.3f}s")
stats_tool.record_miss(elapsed)

print("\nCall 2 (same symbol):")
start = time.time()
price2 = fetch_stock_price("AAPL")  # Should hit cache
elapsed = time.time() - start
print(f"  Price: ${price2}")
print(f"  ⏱️  Time: {elapsed:.3f}s")
stats_tool.record_hit(elapsed)

stats_tool.report()

print("\n💡 Key Advantages:")
print("  - 100x+ faster (cache vs API)")
print("  - Saves on API costs")
print("  - Persistent across restarts")
print("  - Automatic TTL expiration")
print(f"\nCache persists in: ./tool_cache directory")

# Advanced Caching Patterns

## Multi-Level Cache (L1 + L2)

Combine fast in-memory cache (L1) with persistent disk cache (L2) for optimal performance:

In [ ]:
Query → Check L1 (RAM) → If hit: return →
        If miss: Check L2 (Disk) → If hit: promote to L1 & return →
        If miss: Call LLM, store in both L1 and L2

**Benefits:**

- L1: Fastest access (milliseconds)

- L2: Persistence across restarts

- Automatic promotion of hot data to L1

---

## Cache Invalidation Strategies

In [ ]:
### 1. Time-Based (TTL)
cache.set(key, value, expire=3600)  # 1 hour

**Best for**: Time-sensitive data (weather, news)

In [ ]:
### 2. Size-Based (LRU)
@lru_cache(maxsize=128)  # Keep 128 most recent

**Best for**: Memory-constrained environments

In [ ]:
### 3. Manual Invalidation
cache.clear()  # Clear all
del cache[key]  # Remove specific

**Best for**: Data updates, configuration changes

In [ ]:
### 4. Version-Based
cache_key = f"v2:{query}"  # Bump version when schema changes

**Best for**: Schema/format changes

---

## Production Scaling: Redis Cache

For **multi-server** or **high-throughput** applications:

In [ ]:
from langchain.cache import RedisCache
import redis

redis_client = redis.Redis(host='localhost', port=6379)
set_llm_cache(RedisCache(redis_client))

**When to use Redis:**

- Multiple application servers

- Distributed systems

- Very high request rates (>1000/sec)

- Need cache sharing across services

---

## Cache Monitoring

Track cache performance in production:

In [ ]:
def log_cache_metrics(stats):
    if stats.hit_rate < 0.30:
        print("⚠️  Low hit rate - review cache strategy")
    
    if stats.cost_saved > 100:
        print(f"💰 Excellent! Saved ${stats.cost_saved:.2f}")

**Key metrics:**

- Hit rate (target: >50%)

- Response time improvement

- Cost savings

- Cache size growth

In [ ]:
# Multi-Level Cache Implementation
class MultiLevelCache:
    """Two-level cache: L1 (memory) + L2 (disk)"""
    def __init__(self, l1_size=100):
        self.l1_cache = {}  # Fast: In-memory
        self.l1_maxsize = l1_size
        self.l2_cache = Cache("./ml_cache")  # Persistent: Disk
    
    def get(self, key):
        # Try L1 first (fastest)
        if key in self.l1_cache:
            print(f"✅ L1 HIT (RAM)")
            return self.l1_cache[key]
        
        # Try L2 (persistent)
        if key in self.l2_cache:
            value = self.l2_cache[key]
            print(f"✅ L2 HIT (Disk) - Promoting to L1")
            # Promote to L1
            self._add_to_l1(key, value)
            return value
        
        print(f"❌ MISS (both levels)")
        return None
    
    def set(self, key, value, expire=3600):
        """Store in both L1 and L2"""
        # Add to L1
        self._add_to_l1(key, value)
        # Add to L2 with TTL
        self.l2_cache.set(key, value, expire=expire)
        print(f"💾 Stored in L1 + L2")
    
    def _add_to_l1(self, key, value):
        """Add to L1 with size limit (LRU)"""
        if len(self.l1_cache) >= self.l1_maxsize:
            # Remove oldest
            oldest = next(iter(self.l1_cache))
            del self.l1_cache[oldest]
        self.l1_cache[key] = value

# Initialize multi-level cache
ml_cache = MultiLevelCache(l1_size=10)

print("✅ Multi-Level Cache ready!")
print(f"L1 (RAM): max {ml_cache.l1_maxsize} entries")
print(f"L2 (Disk): ./ml_cache directory")

# Demo
print("\n--- Demo ---")
ml_cache.set("test_key", "test_value")
ml_cache.get("test_key")  # L1 hit
ml_cache.l1_cache.clear()  # Simulate L1 eviction
ml_cache.get("test_key")  # L2 hit + promote to L1

# Best Practices

## 1. Cache Key Design

In [ ]:
❌ **Wrong**: Use only prompt text
key = prompt  # Ignores temperature, max_tokens, etc.

In [ ]:
✅ **Right**: Include all relevant parameters
import hashlib
import json

def create_cache_key(prompt, model, temperature, max_tokens):
    key_dict = {
        "prompt": prompt,
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    key_str = json.dumps(key_dict, sort_keys=True)
    return hashlib.md5(key_str.encode()).hexdigest()

---

## 2. Temperature Setting for Caching

In [ ]:
❌ **Wrong**: Cache with temperature > 0 (non-deterministic)
llm = ChatOpenAI(temperature=0.7)  # Different responses each time

In [ ]:
✅ **Right**: Use temperature=0 for exact caching
llm = ChatOpenAI(temperature=0)  # Deterministic responses
# OR use semantic cache (handles variations)

---

## 3. Cache Size Management

In [ ]:
❌ **Wrong**: Unbounded cache growth
cache = {}  # Grows forever → memory leak

In [ ]:
✅ **Right**: Use size limits
@lru_cache(maxsize=128)  # Keep only 128 most recent
# OR use TTL
cache.set(key, value, expire=3600)  # Auto-expire after 1 hour

---

## 4. TTL Strategy
| Data Type | TTL | Rationale |

|-----------|-----|-----------|

| Definitions | 7 days | Static knowledge |

| Weather | 1 hour | Changes gradually |

| News | 15 min | Updates frequently |

| Stock prices | 1 min | Real-time data |

| Live scores | Don't cache | Constantly changing |

---

## 5. Error Handling

In [ ]:
✅ **Always handle cache failures gracefully**:
try:
    cached = cache.get(key)
    if cached:
        return cached
except Exception as e:
    print(f"Cache error: {e}")
    # Fall through to LLM call

# Proceed with LLM call
return llm.invoke(query)

---

## 6. Common Pitfalls

**Pitfall 1**: Caching non-deterministic outputs

- **Solution**: Use temperature=0 or semantic cache

**Pitfall 2**: No cache expiration

- **Solution**: Always set TTL for time-sensitive data

**Pitfall 3**: Ignoring cache hit rate

- **Solution**: Monitor metrics, adjust strategy if <30%

**Pitfall 4**: Caching sensitive data

- **Solution**: Never cache PII, use encryption if needed

# Decision Tree: Which Caching Technique?

## Follow this flowchart to select the right caching approach:

In [ ]:
START: What are you caching?

├─ LLM RESPONSES?
│  ├─ Need persistence? 
│  │  ├─ YES → SQLite Cache (production)
│  │  └─ NO → In-Memory or LangChain InMemory (dev/test)
│  │
│  ├─ Queries vary in wording?
│  │  └─ YES → Semantic Cache (similarity-based)
│  │
│  └─ Large repeated context?
│      └─ YES → Prompt Caching (provider-level, 90% savings)

├─ EXTERNAL API/TOOL RESULTS?
│  └─ → Tool Result Cache with DiskCache
│     ├─ Set TTL based on data freshness
│     ├─ Weather: 1 hour
│     ├─ Stock: 1-5 minutes
│     └─ Definitions: 24+ hours

├─ MULTIPLE SERVERS?
│  └─ → Redis Cache (distributed, shared)

└─ PRODUCTION AT SCALE?
   └─ → Multi-Level Cache (L1: RAM + L2: Disk/Redis)

---

## Quick Selection Matrix

| **Your Scenario** | **Recommended Technique** | **Why** |

|-------------------|--------------------------|---------|

| Just starting, testing | In-Memory Dict | Simplest, zero setup |

| Using LangChain | LangChain InMemoryCache | One-line integration |

| Q&A bot, paraphrased queries | Semantic Cache | Matches similar meanings |

| Production single server | SQLite Cache | Persistence + reliability |

| RAG with large docs | Prompt Caching | 50-90% cost reduction |

| External APIs | Tool Result Cache | Saves API calls |

| Multiple servers | Redis Cache | Distributed, shared |

| High traffic + persistence | Multi-Level (RAM + Disk) | Best of both worlds |

---

## Combining Techniques

You can (and should!) use multiple caching techniques:

**Example: Production RAG System**

- **Prompt Caching**: For large document context (90% cost savings)

- **Semantic Cache**: For user query variations (30-50% hit rate)

- **Tool Cache**: For external API calls (100% savings on duplicates)

- **SQLite**: As persistent layer for all caches

**Result**: 70-85% total cost reduction!

# Summary & Cheat Sheet

## Caching Techniques at a Glance

| Technique | Setup | Persistence | Use Case | Savings |

|-----------|-------|-------------|----------|---------|

| **In-Memory Dict** | `cache = {}` | No | Dev/testing | 100% on duplicates |

| **LangChain InMemory** | `set_llm_cache(InMemoryCache())` | No | Quick wins | High |

| **Semantic** | Custom or LangChain | Optional | Q&A, variations | 30-70% |

| **SQLite** | `SQLiteCache("cache.db")` | Yes | Production | High |

| **Prompt Caching** | Provider API | Provider | RAG, large context | 50-90% |

| **Tool Cache** | `@cached_tool(expire=3600)` | Yes | APIs, tools | Very high |

| **Redis** | `RedisCache(client)` | Yes | Multi-server | High |

---

## Key Takeaways

### 1. Start Simple

Begin with **In-Memory** or **LangChain InMemory** for immediate benefits

### 2. Add Persistence

Move to **SQLite** for production single-server apps

### 3. Optimize for Scale

- **Semantic Cache**: Handle query variations

- **Prompt Caching**: Save on large contexts

- **Tool Cache**: Reduce external API calls

### 4. Monitor Performance

Track these metrics:

- **Hit rate** (target: >50%)

- **Response time** improvement

- **Cost savings** ($$$)

### 5. Common Mistakes to Avoid

❌ Caching with temperature > 0  

❌ No TTL on time-sensitive data  

❌ Ignoring cache size limits  

❌ Not monitoring hit rates  

---

## Performance Impact Summary

Based on typical production scenarios:

| Metric | Without Caching | With Caching | Improvement |

|--------|----------------|--------------|-------------|

| **Response Time** | 1-3 seconds | <100ms | **10-30x faster** |

| **API Costs** | $1,000/month | $100-300/month | **70-90% savings** |

| **Hit Rate** | 0% | 50-80% | **50-80% fewer API calls** |

---

## Next Steps

### Explore Related Topics

- **Memory Management**: [agent_memory.ipynb](agent_memory.ipynb) - Conversation state vs caching

- **Session Management**: [checkpoint_session_management.ipynb](checkpoint_session_management.ipynb) - Production state persistence

### Further Learning

- **LangChain Docs**: [https://python.langchain.com/docs/modules/model_io/llms/llm_caching](https://python.langchain.com/docs/modules/model_io/llms/llm_caching)

- **Anthropic Prompt Caching**: [https://docs.anthropic.com/claude/docs/prompt-caching](https://docs.anthropic.com/claude/docs/prompt-caching)

- **Redis Caching Guide**: [https://redis.io/docs/manual/patterns/](https://redis.io/docs/manual/patterns/)

---

## Quick Reference Commands

In [ ]:
# In-Memory
cache = {}
cache[key] = value

# LangChain
from langchain.cache import InMemoryCache, SQLiteCache
set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache("cache.db"))

# Tool Caching
from diskcache import Cache
tool_cache = Cache("./cache")
tool_cache.set(key, value, expire=3600)

# Clear Cache
set_llm_cache(None)  # Disable
cache.clear()  # Empty

# Conclusion

## What You've Learned

Congratulations! You now understand **6 essential caching techniques** for LLM/Agent systems:

1. ✅ **In-Memory Caching** - Simplest approach with Python dicts

2. ✅ **LangChain InMemory** - Framework-integrated automatic caching

3. ✅ **Semantic Caching** - Similarity-based matching for query variations

4. ✅ **SQLite Persistent** - Production-ready disk caching

5. ✅ **Prompt Caching** - Provider-level for 50-90% cost savings

6. ✅ **Tool Result Caching** - External API optimization

**Plus**: Multi-level caching, best practices, and decision frameworks

---

## Implementation Checklist

When adding caching to your project:

- [ ] Identify what to cache (responses, tools, embeddings)

- [ ] Choose appropriate technique(s) from decision tree

- [ ] Set proper TTL based on data freshness requirements

- [ ] Use temperature=0 for deterministic caching

- [ ] Implement cache size limits (LRU or TTL)

- [ ] Add monitoring (hit rate, cost savings)

- [ ] Handle cache failures gracefully

- [ ] Document cache strategy for your team

---

## Real-World Impact

**Typical production benefits:**

- 💰 **70-90% cost reduction** on LLM API bills

- ⚡ **10-100x faster** response times

- 📊 **50-80% fewer** API calls

- 🛡️ **Better reliability** with offline fallback

---

## Remember the Distinction

**Caching** (this notebook):

- Purpose: Performance & cost optimization

- Stores: LLM responses, API results, computations

- Duration: Temporary with TTL

- Goal: Avoid redundant expensive operations

**Memory** ([agent_memory.ipynb](agent_memory.ipynb)):

- Purpose: Conversation context preservation

- Stores: Messages, user state, history

- Duration: Session or persistent

- Goal: Maintain continuity and context

**Use both together** for optimal production systems!

---

## Final Tips

1. **Start simple**: Begin with In-Memory or LangChain cache

2. **Measure impact**: Track hit rates and cost savings

3. **Iterate**: Adjust TTL and thresholds based on metrics

4. **Combine techniques**: Use multiple caching layers

5. **Monitor production**: Set alerts for low hit rates

---

**Happy caching!** 🚀

*For questions or feedback, refer to LangChain documentation or the related notebooks in this directory.*